# Run the full workflow

In [1]:
import datetime

from st.dto.data import PriceDataDTO, ReturnsDTO, CorrelationDTO
from st.dto.volatility import StandardVolatilityDTO, EWMAVolatilityDTO, RobustVolatilityDTO, \
    VolatilityStandardizationDTO
from st.plotter import PriceDataPlotter, ReturnsPlotter, VolatilityPlotter  # noqa

## Price Data, Returns, Correlation Matrix---

In [2]:
# load price data
tickers = ["AAPL", "GOOGL", "AMZN", "MSFT"]
price_datas = [PriceDataDTO(ticker=tkr, start_date=datetime.datetime(year=2023, month=1, day=1)) for tkr in tickers]

# calculate returns
returns = [ReturnsDTO(price_data=pdata) for pdata in price_datas]

# calculate correlations
correlations = CorrelationDTO(price_datas=price_datas)

In [3]:
for pdata in price_datas: print(pdata)
for rt in returns: print(rt)
print(correlations)
correlations.correlation_matrix

PriceData(ticker=AAPL, start_date=2023-01-01 00:00:00, end_date=2025-12-17, interval=1d, shape=(742, 6))
PriceData(ticker=GOOGL, start_date=2023-01-01 00:00:00, end_date=2025-12-17, interval=1d, shape=(742, 6))
PriceData(ticker=AMZN, start_date=2023-01-01 00:00:00, end_date=2025-12-17, interval=1d, shape=(742, 6))
PriceData(ticker=MSFT, start_date=2023-01-01 00:00:00, end_date=2025-12-17, interval=1d, shape=(742, 6))
Returns(ticker=AAPL, return_type=log, shape=(742,), skew=0.4981)
Returns(ticker=GOOGL, return_type=log, shape=(742,), skew=-0.0661)
Returns(ticker=AMZN, return_type=log, shape=(742,), skew=0.0718)
Returns(ticker=MSFT, return_type=log, shape=(742,), skew=0.3362)
Correlation(tickers=['AAPL', 'GOOGL', 'AMZN', 'MSFT'], total_observations=742)


,AAPL,GOOGL,AMZN,MSFT
AAPL,1.000000,0.459087,0.466124,0.486819
GOOGL,0.459087,1.000000,0.561691,0.492373
AMZN,0.466124,0.561691,1.000000,0.607032
MSFT,0.486819,0.492373,0.607032,1.000000


## Plotter for PriceData
---

In [4]:
plotter = PriceDataPlotter()
for pdata in price_datas:
    plotter.add(pdata)

print(plotter)
plotter.show()

PriceDataPlotter: title=Price Data Visualization, price_data=dict_keys(['AAPL', 'GOOGL', 'AMZN', 'MSFT'])


## Plotter for returns
---

In [5]:
plotter = ReturnsPlotter()
for rt in returns:
    print(rt)
    plotter.add(rt)
print(plotter)
plotter.show(plot_type="both", width=900, height=600)

Returns(ticker=AAPL, return_type=log, shape=(742,), skew=0.4981)
Returns(ticker=GOOGL, return_type=log, shape=(742,), skew=-0.0661)
Returns(ticker=AMZN, return_type=log, shape=(742,), skew=0.0718)
Returns(ticker=MSFT, return_type=log, shape=(742,), skew=0.3362)
ReturnsPlotter(tickers=['AAPL', 'GOOGL', 'AMZN', 'MSFT'])


In [6]:
plotter.show_distribution()

In [7]:
plotter.show_skew_comparison()

## Volatility

- all the volatilises are similar.
- EWMA has sudden peaks (undesirable)
- Robust gives the lowest values most of the time. (undesirable)
- Standard seems a good one to use for volatility standardization.
---

In [8]:
vols_classes = [StandardVolatilityDTO, EWMAVolatilityDTO, RobustVolatilityDTO]
vols_data = {}
for rt in returns:
    tkr = rt.price_data.ticker
    vols_data[tkr] = [vc(returns=rt) for vc in vols_classes]
    plotter = VolatilityPlotter()
    [plotter.add(i, tkr + i.__class__.__name__) for i in vols_data[tkr]]
    plotter.show()
vols_data

{'AAPL': [StandardVolatilityDTO(ticker=AAPL, shape=(742,)),
  EWMAVolatilityDTO(ticker=AAPL, shape=(742,)),
  RobustVolatilityDTO(ticker=AAPL, shape=(742,))],
 'GOOGL': [StandardVolatilityDTO(ticker=GOOGL, shape=(742,)),
  EWMAVolatilityDTO(ticker=GOOGL, shape=(742,)),
  RobustVolatilityDTO(ticker=GOOGL, shape=(742,))],
 'AMZN': [StandardVolatilityDTO(ticker=AMZN, shape=(742,)),
  EWMAVolatilityDTO(ticker=AMZN, shape=(742,)),
  RobustVolatilityDTO(ticker=AMZN, shape=(742,))],
 'MSFT': [StandardVolatilityDTO(ticker=MSFT, shape=(742,)),
  EWMAVolatilityDTO(ticker=MSFT, shape=(742,)),
  RobustVolatilityDTO(ticker=MSFT, shape=(742,))]}

## Volatility Standardization
---

In [9]:
# load price data
tickers = ["AAPL", "GOOGL", "AMZN", "MSFT"]
price_datas = [PriceDataDTO(ticker=tkr, start_date=datetime.datetime(year=2023, month=1, day=1)) for tkr in tickers]

print("Non Standardized Returns")
for rt in returns:
    print(rt)

# normal returns
rts = [ReturnsDTO(price_data=pdata) for pdata in price_datas]
rt_plotter = ReturnsPlotter()
for rt in rts:
    rt_plotter.add(rt)
rt_plotter.show()

print("Standardized Returns")

# volatility standardization
vol_st = [VolatilityStandardizationDTO(price_data=pdata) for pdata in price_datas]
standard_rts = [i.returns for i in vol_st]

plotter = VolatilityPlotter()
for v in vol_st:
    plotter.add(v.volatility)
plotter.show()

# plotting
plotter = ReturnsPlotter()
for rt in standard_rts:
    plotter.add(rt)

# for pdata in price_datas:
#     plotter.add(pdata)

plotter.show()

Non Standardized Returns
Returns(ticker=AAPL, return_type=log, shape=(742,), skew=0.4981)
Returns(ticker=GOOGL, return_type=log, shape=(742,), skew=-0.0661)
Returns(ticker=AMZN, return_type=log, shape=(742,), skew=0.0718)
Returns(ticker=MSFT, return_type=log, shape=(742,), skew=0.3362)


Standardized Returns
